<!-- FULL: keep, DEMO: keep -->
<div style='display:flex; align-items:center; justify-content:space-between; border-bottom:3px solid rgb(255,106,0); padding-bottom:1em;'>
<div>
<span style='color:rgb(22,60,105); font-size:1.8em; font-weight:bold;'>Introduction to Deep Learning</span><br>
<span style='color:rgb(0,85,100); font-size:1.3em;'>Session 6 &mdash; 6: Optimisers &mdash; torch.optim</span><br>
<span style='color:rgb(0,85,100); font-size:1.0em;'>Magda Gregorová &nbsp;·&nbsp; THWS &nbsp;·&nbsp; May 2026</span>
</div>
<img src='../../Common/Pics/thws-logo_vert_en_orange-rgb.png' style='height:80px;'/>
</div>

<!-- FULL: keep, DEMO: delete -->
*We have a working training loop with Adam. Now let's look at all available optimisers and how to configure them.*

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import matplotlib.pyplot as plt
import sys
sys.path.append('../../A2/')
from helpers import load_data

X, y = load_data('../../A2/data.csv')
from plotting import plot_losses, plot_losses_compare

class MLP(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        self.layer1 = nn.Linear(in_features, hidden)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(hidden, out_features)
    def forward(self, x):
        return self.layer2(self.relu(self.layer1(x)))

class DeepMLP(nn.Module):
    def __init__(self, in_features, hidden_sizes, out_features):
        super().__init__()
        sizes  = [in_features] + hidden_sizes
        layers = []
        for i in range(len(sizes) - 1):
            layers.append(nn.Linear(sizes[i], sizes[i+1]))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(sizes[-1], out_features))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

dataset = TensorDataset(X, y)
torch.manual_seed(0)
n_train = int(0.8 * len(dataset))
train_dataset, val_dataset = random_split(dataset, [n_train, len(dataset)-n_train])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

def train_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total = 0.0
    for Xb, yb in loader:
        optimizer.zero_grad(); loss = loss_fn(model(Xb), yb)
        loss.backward(); optimizer.step()
        total += loss.item() * len(Xb)
    return total / len(loader.dataset)

def val_epoch(model, loader, loss_fn):
    model.eval(); total = 0.0
    with torch.no_grad():
        for Xb, yb in loader:
            loss = loss_fn(model(Xb), yb)
            total += loss.item() * len(Xb)
    return total / len(loader.dataset)


<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  <code>torch.optim</code> — overview
</div>

<!-- FULL: keep, DEMO: delete -->
All optimisers from Session 5 are available in `torch.optim`.
Each takes `model.parameters()` and implements the update rule.

Update sequence — always in this order:
1. `optimizer.zero_grad()` &nbsp; clear accumulated gradients
2. `loss.backward()` &nbsp; compute gradients
3. `optimizer.step()` &nbsp; apply update rule

In [ ]:
import torch.optim as optim

# SGD
optimizer_sgd = optim.SGD(mlp.parameters(), lr=0.01)

# SGD with momentum
optimizer_momentum = optim.SGD(mlp.parameters(), lr=0.01, momentum=0.9)

# Adam
optimizer_adam = optim.Adam(mlp.parameters(), lr=1e-3)

print('Optimisers created.')

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(0,85,100); background:rgb(235,245,247); color:rgb(0,85,100); font-size:1.1em; font-weight:bold;'>
  &#8644; Alternatives &mdash; optimiser options
</div>

<!-- FULL: keep, DEMO: delete -->
All optimisers from Session 5 are available. Per-layer learning rates are supported via **param groups**.

In [ ]:
# Standard usage
opt_sgd      = optim.SGD(mlp.parameters(),  lr=0.01)
opt_momentum = optim.SGD(mlp.parameters(),  lr=0.01, momentum=0.9, weight_decay=1e-4)
opt_adam     = optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
opt_adamw    = optim.AdamW(mlp.parameters(), lr=1e-3)  # Adam with decoupled weight decay

# Per-layer learning rates via param groups
opt_perlayer = optim.Adam([
    {'params': mlp.layer1.parameters(), 'lr': 1e-4},
    {'params': mlp.layer2.parameters(), 'lr': 1e-3},
])
print('Optimisers created.')

<!-- FULL: keep, DEMO: shorten -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  One optimiser step — in detail
</div>

In [ ]:
# load data
import sys
sys.path.append('../../A2/')
from helpers import load_data
X, y = load_data('../../A2/data.csv')

torch.manual_seed(0)
mlp   = MLP(in_features=4, hidden=8, out_features=1)
optimizer = optim.Adam(mlp.parameters(), lr=1e-3)
loss_fn   = nn.MSELoss()

# before update
w_before = mlp.layer1.weight.data.clone()
print(f'layer1 weight before: {w_before[0, :3]}')

# one step
optimizer.zero_grad()          # 1. clear gradients
y_pred = mlp(X)                # 2. forward pass
loss   = loss_fn(y_pred, y)    # 3. compute loss
loss.backward()                # 4. backward pass
optimizer.step()               # 5. update parameters

w_after = mlp.layer1.weight.data.clone()
print(f'layer1 weight after:  {w_after[0, :3]}')
print(f'loss: {loss.item():.4f}')

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(0,85,100); background:rgb(235,245,247); color:rgb(0,85,100); font-size:1.1em; font-weight:bold;'>
  &#8644; Alternatives &mdash; loss functions — nn vs functional
</div>

<!-- FULL: keep, DEMO: delete -->
PyTorch provides loss functions in two equivalent forms:

- `nn.MSELoss()` — stateful object, can be stored as an attribute
- `F.mse_loss()` — stateless function from `torch.nn.functional`; more flexible

Both compute the same result. `nn` style is more common in training loops; `F` style is common inside `forward()`.

In [ ]:
import torch.nn.functional as F

y_pred = torch.tensor([[1.5], [2.0], [0.5]])
y_true = torch.tensor([[1.0], [2.5], [0.3]])

# nn style
loss_nn = nn.MSELoss()(y_pred, y_true)

# functional style
loss_f  = F.mse_loss(y_pred, y_true)

print(f'nn.MSELoss:  {loss_nn.item():.4f}')
print(f'F.mse_loss:  {loss_f.item():.4f}')
print(f'Identical:   {torch.allclose(loss_nn, loss_f)}')

<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='margin:3em 0 1.5em 0; padding:0.8em 1.2em; background:rgb(22,60,105); border-radius:6px; display:flex; align-items:center; gap:1em;'>
  <span style='color:rgb(255,106,0); font-size:2em; font-weight:bold; line-height:1;'></span>
  <span style='color:white; font-size:1.4em; font-weight:bold;'>Summary</span>
</div>

<!-- FULL: keep, DEMO: keep -->
<div style='margin:3em 0 1.5em 0; padding:0.8em 1.2em; background:rgb(22,60,105); border-radius:6px; display:flex; align-items:center; gap:1em;'>
  <span style='color:rgb(255,106,0); font-size:2em; font-weight:bold; line-height:1;'></span>
  <span style='color:white; font-size:1.4em; font-weight:bold;'>Summary</span>
</div>

<!-- FULL: keep, DEMO: keep -->
| Concept | PyTorch API |
|---|---|
| Automatic differentiation | `requires_grad`, `.backward()`, `.grad` |
| Network definition | `nn.Module` subclass, `nn.Linear`, `nn.ReLU` |
| Parameter access | `model.parameters()`, `model.named_parameters()` |
| Optimisation | `torch.optim.SGD`, `torch.optim.Adam` |
| Data loading | `Dataset`, `TensorDataset`, `DataLoader` |
| Train/eval modes | `model.train()`, `model.eval()`, `torch.no_grad()` |

<div style='margin-top:1em; color:rgb(0,85,100);'><b>Next:</b> Session 7 — Regularisation &amp; Generalisation</div>